In [1]:
# Run this cell only once
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
# samantha_path = Path('.').absolute().parent.parent.parent
samantha_path = Path('/mnt/bn/ashaw-us/repos/comparison/samantha')
print('Samantha path:', samantha_path)
assert samantha_path.name == 'samantha', "Must set semantha_path to project root."
os.chdir(str(samantha_path))

Samantha path: /mnt/bn/ashaw-us/repos/comparison/samantha


In [ ]:
from IPython.display import Audio
from recipes.musiclm.inference.utils import save_wav, slugify
from recipes.musiclm.datasets.inference import InferenceDataset
from recipes.mulan.dataset.mme import MMEDataset
from recipes.musiclm.lightning.rlhf import SemanticSequenceTrainingModule
import json

/usr/local/lib/python3.9/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.9/dist-packages/torchvision/image.so: undefined symbol: _ZN3c104cuda9SetDeviceEi'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [ ]:
# Save extra requirements, so we don't have to load it again
requires = None
output_dir = Path(f'/mnt/bn/ashaw-us/repos/comparison/samantha/rlhf_results')

## Load prompts

In [ ]:
# # Load from prompt_path
# google_prompts = "/mnt/bn/audio-diffusion/data/google_prompts/mnt/bn/audio-diffusion/data/google_prompts/text_prompt_collection_20230713.csv"
# badcase_prompts = "/mnt/bn/audio-diffusion/data/bad_case_prompts/hanoi_prompts_20230809.csv"
# items = InferenceDataset.from_prompt_path(badcase_prompts).items
# prompt_texts = [item['text'] for item in items]

# Load from dataset
validation_dataset = MMEDataset(name="human_labelled_5k_valid", audio_type="nonvocal", seq_len=400, use_pipe=True)
it = iter(validation_dataset)
items = [item for idx, item in enumerate(it) if idx < 10]
prompt_texts = [item['text'] for item in items]

# Run inference

In [ ]:
ckpt_paths = [
    (0, "/mnt/bn/audio-diffusion/ashaw/logs/semantic_flash_llama_rlhf/mulan_text_human_5k/checkpoints/step=0.ckpt"),
    (500, "/mnt/bn/audio-diffusion/ashaw/logs/semantic_flash_llama_rlhf/mulan_text_human_5k/checkpoints/step=000500.ckpt"),
    (1000, "/mnt/bn/audio-diffusion/ashaw/logs/semantic_flash_llama_rlhf/mulan_text_human_5k/checkpoints/step=001000.ckpt")
]

In [ ]:
# Run inference on all checkpoints
for ckpt_steps, ckpt_path in ckpt_paths:
    # Load
    rlhf_model = SemanticSequenceTrainingModule.load_from_checkpoint(ckpt_path)
    rlhf_model.eval().cuda()

    rlhf_model.extra_params.diffusion_steps = 25
    rlhf_model.extra_params.mulan_force_cfg = 0

    if requires is None:
        rlhf_model.load_required_modules()
        requires = rlhf_model.requires
    else:
        rlhf_model.requires = requires

    # Generate
    wavs, rewards = rlhf_model.generate_audio(prompt_texts, hp=None)
    wavs, rewards = wavs.cpu(), rewards.cpu()

    # Write to Dir
    ckpt_output_dir = output_dir/str(ckpt_steps)

    for i, (wav, prompt_text, reward) in enumerate(zip(wavs, prompt_texts, rewards)):
        os.makedirs(ckpt_output_dir, exist_ok=True)
        fname = f"{i}.r[{float(reward):0.2f}].{slugify(prompt_text)[:64]}"
        wav_fp = os.path.join(ckpt_output_dir, f"{fname}.wav")
        txt_fp = os.path.join(ckpt_output_dir, f"{fname}.txt")
        print(f"[Saving] {wav_fp}")
        save_wav(wav, wav_fp, sr=24000)
        with open(txt_fp, 'w', encoding='utf-8') as f:
            f.write(prompt_text)

    with open(ckpt_output_dir/'reward.txt', 'w', encoding='utf-8') as f:
        f.write(str(rewards.mean()))

In [ ]:
# Covert to mp3 and zip
parent_path = str(output_dir.absolute().parent)
dir_name = output_dir.name

mp3_cmd = f'cd "{output_dir}" ' + '&& find . -name "*.wav" -exec ffmpeg -y -i {} {}.mp3 \; -exec rm {} \; && popd'
os.system(mp3_cmd)

zip_cmd = f'cd "{parent_path}" && zip -r "{dir_name}".zip "{dir_name}"'
os.system(zip_cmd)
print(f'Output path: {parent_path}/{dir_name}.zip')